# Questão 2 – Item a)
## Avaliação de Classificadores com Validação Cruzada 30 × 10-folds

**Referência classificadores**: Kittler et al. (1998) "On Combining Classifiers"  
**Dataset**: Ionosphere (UCI) — 351 amostras, 33 features (coluna constante removida)

**Versões do dataset:**
- **V1**: variável resposta original binária (good=1, bad=0)
- **V2**: variável resposta = labels de cluster do KCM-K-GH com c*=5 (Questão 1)

**Protocolo:**
- 30 repetições de 10-fold CV estratificado (externo)
- Para classificadores com hiperparâmetros (ii, iii, iv): inner 5-fold CV nos 9 folds de treino
- Retreinar com todos os 9 folds usando hiperparâmetros selecionados
- Amostragem estratificada em todas as etapas

**Classificadores:**
- i) Bayesiano Gaussiano (MLE normal multivariada, Σ distinto por classe)
- ii) Bayesiano k-vizinhos (Euclidiana, City-Block, Chebyshev; tunar k e distância)
- iii) Bayesiano Janela de Parzen (kernel produto gaussiano univariado; tunar h)
- iv) Regressão Logística (tunar C)
- v) Voto Majoritário dos classificadores i–iv (Kittler 1998, Eq. 20)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from scipy.special import logsumexp
from itertools import product as iterproduct
import os, warnings
warnings.filterwarnings('ignore')


In [ ]:
# ── Carregamento do dataset Ionosphere ───────────────────────────────────────
ionosphere = fetch_openml(name='ionosphere', version=1, as_frame=True)
X_raw = ionosphere.data.values.astype(float)
y_raw = ionosphere.target.values

# Remover coluna constante (coluna 1, todos zeros)
col_std = X_raw.std(axis=0)
X = X_raw[:, col_std > 1e-10]      # (351, 33)
y_v1 = (y_raw == 'g').astype(int)  # V1: 0=bad, 1=good

print(f"Shape X: {X.shape}")
print(f"V1 – distribuição: {np.bincount(y_v1)}  (0=bad, 1=good)")


In [ ]:
# ── Labels V2: clusters KCM-K-GH c*=5 (da Questão 1) ─────────────────────────
def _compute_kernel(X, g, inv_s2):
    K = np.zeros((len(X), len(g)))
    for k in range(len(g)):
        K[:, k] = np.exp(-0.5 * ((X - g[k])**2 @ inv_s2))
    return K

def kcm_k_gh(X, c, max_iter=300, eps=1e-10, seed=None):
    rng = np.random.default_rng(seed)
    N, P = X.shape
    g = X[rng.choice(N, c, replace=False)].copy()
    inv_s2 = np.ones(P)
    labels = np.argmax(_compute_kernel(X, g, inv_s2), axis=1)
    J_hist = []
    for _ in range(max_iter):
        old = labels.copy()
        # Step 1: protótipos (Eq.14)
        K = _compute_kernel(X, g, inv_s2)
        for k in range(c):
            m = labels == k
            if m.sum() > 0:
                w = K[m, k]; g[k] = (X[m].T @ w) / (w.sum() + eps)
        # Step 2: larguras (Eq.16, γ=1)
        K = _compute_kernel(X, g, inv_s2)
        D = np.zeros(P)
        for k in range(c):
            m = labels == k
            if m.sum() > 0:
                w = K[m, k]; D += ((X[m]-g[k])**2).T @ w
        D = np.maximum(D, eps)
        inv_s2 = np.exp(np.mean(np.log(D)) - np.log(D))
        # Step 3: alocação (Eq.18)
        K = _compute_kernel(X, g, inv_s2)
        labels = np.argmax(K, axis=1)
        J = 2.0 * np.sum(1.0 - K[np.arange(N), labels])
        J_hist.append(J)
        if np.array_equal(labels, old): break
    return g, 1.0/inv_s2, labels, J_hist

if os.path.exists('labels_opt.npy'):
    y_v2 = np.load('labels_opt.npy')
    print(f"labels_opt.npy carregado.")
else:
    print("Recomputando KCM-K-GH c*=5 (100 runs)...")
    best_J, best_labels = np.inf, None
    for run in range(100):
        _, _, lbl, hist = kcm_k_gh(X, c=5, seed=run)
        if hist[-1] < best_J:
            best_J, best_labels = hist[-1], lbl.copy()
    y_v2 = best_labels
    np.save('labels_opt.npy', y_v2)
    print(f"Computado. J={best_J:.4f}")

print(f"V2 – distribuição por cluster: {np.bincount(y_v2)}")


## Implementação dos Classificadores

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# i) CLASSIFICADOR BAYESIANO GAUSSIANO
#    Regra de Bayes com MLE de normal multivariada, Σ distinto por classe
#    p(x|ωi,θi) = (2π)^{-d/2} |Σ̂i^{-1}|^{1/2} exp{-½(x-μ̂i)ᵀΣ̂i^{-1}(x-μ̂i)}
#    P(ωi) = ni/n  (MLE do prior)
# ════════════════════════════════════════════════════════════════════════════
class GaussianBayesClassifier:
    """Classificador Bayesiano Gaussiano (QDA) com estimação MLE."""

    def __init__(self, reg=1e-6):
        self.reg = reg   # regularização para evitar Σ singular

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        N, d = X.shape
        self._params = {}
        for c in self.classes_:
            Xc = X[y == c]
            Nc = len(Xc)
            mu = Xc.mean(axis=0)
            # MLE da covariância: divide por N (não N-1)
            diff = Xc - mu
            Sigma = (diff.T @ diff) / Nc + self.reg * np.eye(d)
            sign, logdet = np.linalg.slogdet(Sigma)
            Sigma_inv = np.linalg.inv(Sigma)
            log_prior = np.log(Nc / N)
            self._params[c] = (mu, Sigma_inv, logdet, log_prior)
        return self

    def predict(self, X):
        """Aplica a regra de decisão bayesiana: argmax_i P(ωi|x)"""
        log_post = np.zeros((len(X), len(self.classes_)))
        for i, c in enumerate(self.classes_):
            mu, S_inv, logdet, log_prior = self._params[c]
            diff = X - mu                               # (N_test, d)
            maha = np.sum(diff @ S_inv * diff, axis=1)  # distância de Mahalanobis²
            # log p(x|ωi) + log P(ωi)  ∝  log P(ωi|x)
            log_post[:, i] = -0.5 * (maha + logdet) + log_prior
        return self.classes_[np.argmax(log_post, axis=1)]

    def get_params(self, deep=True): return {'reg': self.reg}
    def set_params(self, **p):
        for k, v in p.items(): setattr(self, k, v)
        return self


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# iii) CLASSIFICADOR BAYESIANO - JANELA DE PARZEN
#      kernel multivariado produto de kernels gaussianos univariados:
#      p(x|ωi) = (1/Ni) Σ_{xj∈ωi} ∏_d (1/h√2π) exp(-(xd-xjd)²/2h²)
# ════════════════════════════════════════════════════════════════════════════
class ParzenWindowClassifier:
    """Estimador de densidade por janela de Parzen com kernel produto gaussiano."""

    def __init__(self, h=1.0):
        self.h = h

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        N = len(X)
        self._Xc = {}
        self._log_prior = {}
        for c in self.classes_:
            mask = y == c
            self._Xc[c] = X[mask]
            self._log_prior[c] = np.log(mask.sum() / N)
        return self

    def _log_density(self, X_q, Xc):
        """Log-densidade vetorizada para todos os pontos de consulta."""
        h, d = self.h, Xc.shape[1]
        # (Nq, Nc, d) → log kernel por ponto e por padrão de treino
        diff = (X_q[:, np.newaxis, :] - Xc[np.newaxis, :, :]) / h
        log_k = -0.5 * np.sum(diff**2, axis=2) - d * np.log(h * np.sqrt(2*np.pi))
        # soma sobre padrões de treino (log-sum-exp) - log(Nc)
        return logsumexp(log_k, axis=1) - np.log(len(Xc))

    def predict(self, X):
        """Aplica a regra de Bayes: argmax_i [log p(x|ωi) + log P(ωi)]"""
        log_post = np.zeros((len(X), len(self.classes_)))
        for i, c in enumerate(self.classes_):
            log_post[:, i] = self._log_density(X, self._Xc[c]) + self._log_prior[c]
        return self.classes_[np.argmax(log_post, axis=1)]

    def get_params(self, deep=True): return {'h': self.h}
    def set_params(self, **p):
        for k, v in p.items(): setattr(self, k, v)
        return self


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# v) CLASSIFICADOR VOTO MAJORITÁRIO (Kittler et al. 1998, Eq. 20)
#    Δ_{ki} = 1 se classificador i prediz classe k, 0 caso contrário
#    Assign Z → ω_j  se  Σ_i Δ_{ji} = max_k Σ_i Δ_{ki}
# ════════════════════════════════════════════════════════════════════════════
class MajorityVoteClassifier:
    """Classificador por voto majoritário (hard voting) de múltiplos classificadores."""

    def __init__(self, classifiers):
        self.classifiers = classifiers  # lista de classificadores i–iv

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        for clf in self.classifiers:
            clf.fit(X, y)
        return self

    def predict(self, X):
        """Cada classificador vota; classe com mais votos é a predição final."""
        # predictions: (n_classifiers, n_samples)
        votes = np.array([clf.predict(X) for clf in self.classifiers])
        result = np.empty(len(X), dtype=votes.dtype)
        for i in range(len(X)):
            vals, counts = np.unique(votes[:, i], return_counts=True)
            result[i] = vals[np.argmax(counts)]
        return result


## Funções de Tuning de Hiperparâmetros (inner 5-fold CV)

In [ ]:
def _inner_cv_score(clf, X, y, n_folds, avg):
    """F-measure médio em n_folds estratificados."""
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=7)
    scores = []
    for tr, val in skf.split(X, y):
        clf.fit(X[tr], y[tr])
        pred = clf.predict(X[val])
        scores.append(f1_score(y[val], pred, average=avg, zero_division=0))
    return np.mean(scores)

def tune_knn(X_train, y_train, n_inner=5):
    """Tunar k ∈ {1,3,5,...,15} e distância ∈ {euclidean, cityblock, chebyshev}."""
    avg = 'macro' if len(np.unique(y_train)) > 2 else 'binary'
    best_score, best_params = -np.inf, {'n_neighbors': 1, 'metric': 'euclidean'}
    # Limitar k ao tamanho do fold de treino no inner CV para evitar n_neighbors > n_samples_fit
    max_k = max(1, int(len(X_train) * (n_inner - 1) / n_inner) - 1)
    k_candidates = [k for k in [1, 3, 5, 7, 9, 11, 13, 15] if k <= max_k] or [1]
    for k, m in iterproduct(k_candidates,
                             ['euclidean', 'cityblock', 'chebyshev']):
        clf = KNeighborsClassifier(n_neighbors=k, metric=m)
        score = _inner_cv_score(clf, X_train, y_train, n_inner, avg)
        if score > best_score:
            best_score = score
            best_params = {'n_neighbors': k, 'metric': m}
    return best_params

def tune_parzen(X_train, y_train, n_inner=5):
    """Tunar h ∈ {0.01, 0.05, 0.1, 0.3, 0.5, 1.0, 2.0, 5.0}."""
    avg = 'macro' if len(np.unique(y_train)) > 2 else 'binary'
    best_score, best_h = -np.inf, 1.0
    for h in [0.01, 0.05, 0.1, 0.3, 0.5, 1.0, 2.0, 5.0]:
        clf = ParzenWindowClassifier(h=h)
        score = _inner_cv_score(clf, X_train, y_train, n_inner, avg)
        if score > best_score:
            best_score, best_h = score, h
    return {'h': best_h}

def tune_logreg(X_train, y_train, n_inner=5):
    """Tunar C ∈ {0.001, 0.01, 0.1, 1, 10, 100}."""
    avg  = 'macro' if len(np.unique(y_train)) > 2 else 'binary'
    mc   = 'multinomial' if len(np.unique(y_train)) > 2 else 'auto'
    best_score, best_C = -np.inf, 1.0
    for C in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]:
        clf = LogisticRegression(C=C, multi_class=mc, solver='lbfgs', max_iter=1000)
        score = _inner_cv_score(clf, X_train, y_train, n_inner, avg)
        if score > best_score:
            best_score, best_C = score, C
    return {'C': best_C}


In [ ]:
def compute_metrics(y_true, y_pred, avg):
    """Retorna dict com taxa de erro, precisão, cobertura e F-measure."""
    return {
        'taxa_erro':  1.0 - accuracy_score(y_true, y_pred),
        'precisao':   precision_score(y_true, y_pred, average=avg, zero_division=0),
        'cobertura':  recall_score(y_true, y_pred, average=avg, zero_division=0),
        'f_measure':  f1_score(y_true, y_pred, average=avg, zero_division=0),
    }

CLF_NAMES = ['GaussianoBayes', 'kNNBayes', 'Parzen', 'RegLogistica', 'VotoMajoritario']
METRICAS  = ['taxa_erro', 'precisao', 'cobertura', 'f_measure']


## Experimento Principal – 30 × 10-fold CV Estratificado

Protocolo por fold:
1. Separar 1 fold de teste e 9 folds de treino (estratificado)
2. Inner 5-fold CV nos 9 folds de treino → selecionar hiperparâmetros (ii, iii, iv)
3. Retreinar **todos os 5 classificadores** nos 9 folds de treino completos
4. Avaliar no fold de teste → registrar métricas

Resultado: para cada repetição, média das 10 métricas de fold → 1 valor por (classificador, métrica, rep)


In [ ]:
def run_experiment(X, y, n_reps=30, n_outer=10, n_inner=5,
                   random_state_base=0, label=''):
    """
    Retorna dict results[clf_name][metrica] = lista com n_reps valores
    (cada valor = média das 10 métricas de fold daquela repetição).
    """
    n_classes = len(np.unique(y))
    avg = 'macro' if n_classes > 2 else 'binary'
    mc  = 'multinomial' if n_classes > 2 else 'auto'

    results = {clf: {m: [] for m in METRICAS} for clf in CLF_NAMES}

    for rep in range(n_reps):
        print(f'  [{label}] rep {rep+1:2d}/{n_reps}', end='\r')

        skf_outer = StratifiedKFold(n_splits=n_outer, shuffle=True,
                                    random_state=random_state_base + rep)

        fold_buf = {clf: {m: [] for m in METRICAS} for clf in CLF_NAMES}

        for train_idx, test_idx in skf_outer.split(X, y):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            # ── Tuning: inner 5-fold nos 9 folds de treino ───────────────
            knn_p = tune_knn(X_train, y_train, n_inner)
            par_p = tune_parzen(X_train, y_train, n_inner)
            lr_p  = tune_logreg(X_train, y_train, n_inner)

            # ── Instanciar classificadores com hiperparâmetros selecionados
            gb  = GaussianBayesClassifier()
            knn = KNeighborsClassifier(**knn_p)
            par = ParzenWindowClassifier(**par_p)
            lr  = LogisticRegression(**lr_p, multi_class=mc,
                                     solver='lbfgs', max_iter=1000)
            # Voto majoritário: instâncias NOVAS com os mesmos hiperparâmetros
            mv  = MajorityVoteClassifier([
                GaussianBayesClassifier(),
                KNeighborsClassifier(**knn_p),
                ParzenWindowClassifier(**par_p),
                LogisticRegression(**lr_p, multi_class=mc,
                                   solver='lbfgs', max_iter=1000),
            ])

            # ── Treinar todos os classificadores nos 9 folds completos ───
            clfs = dict(zip(CLF_NAMES, [gb, knn, par, lr, mv]))
            for name, clf in clfs.items():
                clf.fit(X_train, y_train)
                m = compute_metrics(y_test, clf.predict(X_test), avg)
                for k, v in m.items():
                    fold_buf[name][k].append(v)

        # ── Média das 10 métricas de fold para esta repetição ─────────────
        for clf in CLF_NAMES:
            for m in METRICAS:
                results[clf][m].append(np.mean(fold_buf[clf][m]))

    print(f'\n  [{label}] concluído.')
    return results


In [ ]:
print('Executando experimento V1 (labels originais binários)...')
results_v1 = run_experiment(X, y_v1, n_reps=30, n_outer=10, n_inner=5,
                             random_state_base=0, label='V1')

print('Executando experimento V2 (labels cluster c*=5)...')
results_v2 = run_experiment(X, y_v2, n_reps=30, n_outer=10, n_inner=5,
                             random_state_base=100, label='V2')

print('Experimento item a) concluído.')


## Visualização dos Resultados Brutos (médias das 30 repetições)

In [ ]:
print('=== Médias globais por classificador e métrica ===\n')

for vname, res in [('V1 (labels originais)', results_v1),
                   ('V2 (clusters c*=5)',     results_v2)]:
    print(f'--- {vname} ---')
    header = f"{'Classificador':<22}" + ''.join(f'{m:>14}' for m in METRICAS)
    print(header)
    print('-' * (22 + 14*4))
    for clf in CLF_NAMES:
        row = f'{clf:<22}'
        for m in METRICAS:
            row += f"{np.mean(res[clf][m]):>14.4f}"
        print(row)
    print()


---
## Item b) – Estimativa Pontual e Intervalo de Confiança (t-Student 95%)

Para cada classificador e cada métrica, calcula-se:
- **Estimativa pontual**: média das 30 repetições
- **IC 95%**: intervalo t-Student com n−1=29 graus de liberdade

$$\bar{x} \pm t_{0.025,29} \cdot \frac{s}{\sqrt{30}}$$


In [ ]:
from scipy.stats import t as t_dist

def ci95(values):
    """Retorna (média, limite_inf, limite_sup) com IC t-Student 95%."""
    n   = len(values)
    mu  = np.mean(values)
    se  = np.std(values, ddof=1) / np.sqrt(n)
    t_c = t_dist.ppf(0.975, df=n - 1)
    return mu, mu - t_c * se, mu + t_c * se

print('=== ITEM b) – Estimativas Pontuais e ICs 95% (t-Student) ===\n')

for vname, res in [('V1 – labels originais', results_v1),
                   ('V2 – clusters c*=5',    results_v2)]:
    print(f'{'─'*72}')
    print(f'  {vname}')
    print(f'{'─'*72}')
    for metrica in METRICAS:
        print(f'\n  {metrica.upper()}:')
        print(f'  {"Classificador":<22} {"Média":>8} {"IC inf":>10} {"IC sup":>10}')
        print(f'  {"-"*52}')
        for clf in CLF_NAMES:
            mu, lo, hi = ci95(res[clf][metrica])
            print(f'  {clf:<22} {mu:>8.4f} {lo:>10.4f} {hi:>10.4f}')
    print()


In [ ]:
# Tabela resumida como DataFrame
rows = []
for vname, res in [('V1', results_v1), ('V2', results_v2)]:
    for clf in CLF_NAMES:
        row = {'Versão': vname, 'Classificador': clf}
        for m in METRICAS:
            mu, lo, hi = ci95(res[clf][m])
            row[m] = f'{mu:.4f} [{lo:.4f}, {hi:.4f}]'
        rows.append(row)

df_ic = pd.DataFrame(rows)
print(df_ic.to_string(index=False))


---
## Item b) – Teste de Friedman + Pós-teste de Nemenyi

**Referência**: Demšar (2006) "Statistical Comparisons of Classifiers over Multiple Data Sets"

- **Teste de Friedman** (Iman-Davenport): verifica se existe diferença significativa entre os k=5 classificadores  
  usando as N=30 repetições como "data sets independentes"
- **Pós-teste de Nemenyi**: se Friedman rejeita H₀, identifica quais pares diferem significativamente

$$F_F = \frac{(N-1)\chi^2_F}{N(k-1) - \chi^2_F}, \quad
CD = q_\alpha\sqrt{\frac{k(k+1)}{6N}}$$


In [ ]:
from scipy.stats import f as f_dist
import matplotlib.pyplot as plt

# Valores críticos q_α para Nemenyi (Demšar 2006, Tabela 5a)
Q_ALPHA = {
    2: {0.05: 1.960, 0.10: 1.645},
    3: {0.05: 2.343, 0.10: 2.052},
    4: {0.05: 2.569, 0.10: 2.291},
    5: {0.05: 2.728, 0.10: 2.459},
    6: {0.05: 2.850, 0.10: 2.589},
}

def friedman_nemenyi(results, metrica, alpha=0.05):
    """
    Aplica o teste de Friedman (Iman-Davenport) e calcula a CD de Nemenyi.
    N = número de repetições (30), k = número de classificadores (5).
    """
    k = len(CLF_NAMES)
    N = len(results[CLF_NAMES[0]][metrica])   # 30 repetições

    # Matriz de scores (N × k)
    scores = np.array([results[clf][metrica] for clf in CLF_NAMES]).T  # (30, 5)

    # Rank por linha: rank 1 = melhor desempenho
    reverse = (metrica == 'taxa_erro')   # para taxa de erro, menor é melhor
    ranks = np.zeros_like(scores)
    for i in range(N):
        row = scores[i]
        order = np.argsort(row) if reverse else np.argsort(row)[::-1]
        j = 0
        while j < k:
            j2 = j + 1
            while j2 < k and row[order[j2]] == row[order[j]]: j2 += 1
            avg_r = np.mean(np.arange(j + 1, j2 + 1))
            for idx in range(j, j2): ranks[i, order[idx]] = avg_r
            j = j2

    R_j = ranks.mean(axis=0)   # rank médio por classificador

    # Estatística de Friedman χ²_F e Iman-Davenport F_F
    chi2_F = 12 * N / (k * (k + 1)) * (np.sum(R_j**2) - k * (k + 1)**2 / 4)
    F_F    = (N - 1) * chi2_F / (N * (k - 1) - chi2_F)
    p_val  = 1 - f_dist.cdf(F_F, k - 1, (k - 1) * (N - 1))

    # Diferença crítica de Nemenyi
    CD = Q_ALPHA[k][alpha] * np.sqrt(k * (k + 1) / (6 * N))

    return R_j, F_F, p_val, CD


print('=== TESTE DE FRIEDMAN + PÓS-TESTE DE NEMENYI (α=0.05) ===\n')
print(f'N = 30 repetições,  k = {len(CLF_NAMES)} classificadores\n')

friedman_results = {}
for vname, res in [('V1', results_v1), ('V2', results_v2)]:
    print(f'{'─'*68}')
    print(f'  {vname}')
    print(f'{'─'*68}')
    friedman_results[vname] = {}
    for metrica in METRICAS:
        R_j, F_F, p_val, CD = friedman_nemenyi(res, metrica)
        sig = '*** SIGNIFICATIVO' if p_val < 0.05 else '(não significativo)'
        print(f'\n  {metrica.upper():14s}: F_F={F_F:.4f}, p={p_val:.4f}  {sig}')
        print(f'  Ranks médios:')
        for name, r in zip(CLF_NAMES, R_j):
            print(f'    {name:<22}: {r:.3f}')
        friedman_results[vname][metrica] = (R_j, F_F, p_val, CD)

        if p_val < 0.05:
            print(f'  CD (Nemenyi) = {CD:.4f}')
            found = False
            for i in range(len(CLF_NAMES)):
                for j in range(i + 1, len(CLF_NAMES)):
                    diff = abs(R_j[i] - R_j[j])
                    if diff > CD:
                        print(f'  *** {CLF_NAMES[i]} vs {CLF_NAMES[j]}: '
                              f'|{R_j[i]:.3f} − {R_j[j]:.3f}| = {diff:.3f} > CD')
                        found = True
            if not found:
                print('  (pós-teste não encontrou pares significativos)')
    print()


In [ ]:
def plot_cd_diagram(R_j, CD, clf_names, title='', ax=None):
    """Diagrama de diferença crítica (Demšar 2006, Fig.1)."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 2.5))
    k = len(clf_names)
    order = np.argsort(R_j)           # melhor rank (menor número) à direita
    rnames = [clf_names[i] for i in order]
    rvals  = R_j[order]

    ax.set_xlim([0.5, k + 0.5])
    ax.set_ylim([-1.2, 2.0])
    ax.set_yticks([])
    ax.set_xlabel('Rank médio (menor = melhor)', fontsize=9)
    ax.set_title(title, fontsize=9)
    ax.axhline(1, color='black', lw=1.5)

    for idx, (r, n) in enumerate(zip(rvals, rnames)):
        ax.plot(r, 1, 'ko', ms=5)
        offset = 1.30 if idx % 2 == 0 else 0.65
        ax.text(r, offset, n, ha='center', fontsize=7.5, rotation=30)

    # Barra CD sobre o melhor classificador
    ax.annotate('', xy=(rvals[0] + CD, 1.75), xytext=(rvals[0], 1.75),
                arrowprops=dict(arrowstyle='<->', color='red', lw=1.5))
    ax.text((2 * rvals[0] + CD) / 2, 1.88,
            f'CD={CD:.3f}', ha='center', color='red', fontsize=8)

    # Linhas horizontais agrupando pares não-significativos
    drawn = set()
    for i in range(k):
        for j in range(i + 1, k):
            if abs(rvals[i] - rvals[j]) <= CD and (i, j) not in drawn:
                y_bar = -0.35 - 0.22 * (i % 2)
                ax.plot([rvals[i], rvals[j]], [y_bar, y_bar], 'b-', lw=2.5, alpha=0.7)
                drawn.add((i, j))

    ax.invert_xaxis()
    return ax


# Plotar diagramas CD para F-measure (V1 e V2)
fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
for ax, (vname, res) in zip(axes, [('V1', results_v1), ('V2', results_v2)]):
    R_j, F_F, p_val, CD = friedman_results[vname]['f_measure']
    title = (f'Diagrama CD – F-measure – {vname}\n'
             f'F_F={F_F:.3f}, p={p_val:.4f}'
             + (' ***' if p_val < 0.05 else ''))
    plot_cd_diagram(R_j, CD, CLF_NAMES, title=title, ax=ax)
plt.tight_layout()
plt.savefig('cd_diagram_fmeasure.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva: cd_diagram_fmeasure.png')


---
## Item b) – Curvas de Aprendizado (F-measure)

Para cada fração de treinamento f ∈ {5%, 10%, ..., 95%}:
- Usar subamostra estratificada de tamanho f × |treino| do conjunto de treino
- Tunar hiperparâmetros (inner 5-fold) na subamostra
- Treinar os 5 classificadores na subamostra
- Avaliar F-measure **no conjunto de treino (subamostra)** e **no conjunto de teste**

> Avaliação em treino e teste permite analisar overfitting/underfitting.


In [ ]:
FRACTIONS = np.arange(0.05, 1.00, 0.05)   # 5% a 95%, passo 5%

def learning_curves(X, y, n_outer=10, n_inner=3, rs=0, label=''):
    """
    Para cada fração, faz 1 rodada de 10-fold CV estratificado.
    Retorna: lc_train, lc_test — dicts clf → lista de f1 por fração.
    """
    n_classes = len(np.unique(y))
    avg = 'macro' if n_classes > 2 else 'binary'
    mc  = 'multinomial' if n_classes > 2 else 'auto'

    lc_train = {clf: [] for clf in CLF_NAMES}
    lc_test  = {clf: [] for clf in CLF_NAMES}

    skf = StratifiedKFold(n_splits=n_outer, shuffle=True, random_state=rs)
    splits = list(skf.split(X, y))

    for frac in FRACTIONS:
        fold_train = {clf: [] for clf in CLF_NAMES}
        fold_test  = {clf: [] for clf in CLF_NAMES}

        for train_idx, test_idx in splits:
            X_tr_full, y_tr_full = X[train_idx], y[train_idx]
            X_te, y_te           = X[test_idx],  y[test_idx]

            # Subamostra estratificada do conjunto de treino
            n_sub = max(n_classes * 2, int(round(frac * len(train_idx))))
            if n_sub >= len(train_idx):
                X_sub, y_sub = X_tr_full, y_tr_full
            else:
                try:
                    X_sub, _, y_sub, _ = train_test_split(
                        X_tr_full, y_tr_full,
                        train_size=n_sub, stratify=y_tr_full,
                        random_state=rs)
                except ValueError:
                    X_sub, y_sub = X_tr_full, y_tr_full

            if len(np.unique(y_sub)) < n_classes:
                for clf in CLF_NAMES:
                    fold_train[clf].append(np.nan)
                    fold_test[clf].append(np.nan)
                continue

            # Tuning inner CV na subamostra
            n_folds_inner = min(n_inner, min(np.bincount(y_sub.astype(int))))
            n_folds_inner = max(2, n_folds_inner)
            knn_p = tune_knn(X_sub, y_sub, n_folds_inner)
            par_p = tune_parzen(X_sub, y_sub, n_folds_inner)
            lr_p  = tune_logreg(X_sub, y_sub, n_folds_inner)

            # Instanciar e treinar
            gb  = GaussianBayesClassifier()
            knn = KNeighborsClassifier(**knn_p)
            par = ParzenWindowClassifier(**par_p)
            lr  = LogisticRegression(**lr_p, multi_class=mc,
                                     solver='lbfgs', max_iter=1000)
            mv  = MajorityVoteClassifier([
                GaussianBayesClassifier(),
                KNeighborsClassifier(**knn_p),
                ParzenWindowClassifier(**par_p),
                LogisticRegression(**lr_p, multi_class=mc,
                                   solver='lbfgs', max_iter=1000),
            ])

            clfs = dict(zip(CLF_NAMES, [gb, knn, par, lr, mv]))
            for name, clf in clfs.items():
                clf.fit(X_sub, y_sub)
                # F-measure no treino (subamostra)
                fold_train[name].append(
                    f1_score(y_sub, clf.predict(X_sub), average=avg, zero_division=0))
                # F-measure no teste
                fold_test[name].append(
                    f1_score(y_te, clf.predict(X_te), average=avg, zero_division=0))

        # Média sobre os 10 folds para esta fração
        for clf in CLF_NAMES:
            vtr = [v for v in fold_train[clf] if not np.isnan(v)]
            vte = [v for v in fold_test[clf]  if not np.isnan(v)]
            lc_train[clf].append(np.mean(vtr) if vtr else np.nan)
            lc_test[clf].append(np.mean(vte)  if vte else np.nan)

        print(f'  [{label}] fração={frac:.2f}', end='\r')

    print(f'\n  [{label}] curvas concluídas.')
    return lc_train, lc_test


print('Calculando curvas de aprendizado V1...')
lc_v1_train, lc_v1_test = learning_curves(X, y_v1, rs=200, label='V1')

print('Calculando curvas de aprendizado V2...')
lc_v2_train, lc_v2_test = learning_curves(X, y_v2, rs=300, label='V2')


In [ ]:
COLORS = ['tab:blue', 'tab:red', 'tab:green', 'tab:orange', 'tab:purple']
STYLES = ['-', '--', '-.', ':', (0, (3, 1, 1, 1))]
PCT = FRACTIONS * 100

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

datasets = [
    ('V1 – Labels Originais', lc_v1_train, lc_v1_test),
    ('V2 – Clusters KCM-K-GH (c*=5)', lc_v2_train, lc_v2_test),
]

for col, (vname, lc_tr, lc_te) in enumerate(datasets):
    # Curva no TREINO
    ax = axes[0, col]
    for clf, col_c, ls in zip(CLF_NAMES, COLORS, STYLES):
        ax.plot(PCT, lc_tr[clf], label=clf, color=col_c,
                linestyle=ls, marker='o', markersize=3)
    ax.set_title(f'Treino – {vname}', fontsize=10)
    ax.set_xlabel('Tamanho do treino (%)')
    ax.set_ylabel('F-measure')
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(True, alpha=0.3)
    ax.set_xlim([5, 95]); ax.set_ylim([0, 1.05])

    # Curva no TESTE
    ax = axes[1, col]
    for clf, col_c, ls in zip(CLF_NAMES, COLORS, STYLES):
        ax.plot(PCT, lc_te[clf], label=clf, color=col_c,
                linestyle=ls, marker='o', markersize=3)
    ax.set_title(f'Teste – {vname}', fontsize=10)
    ax.set_xlabel('Tamanho do treino (%)')
    ax.set_ylabel('F-measure')
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(True, alpha=0.3)
    ax.set_xlim([5, 95]); ax.set_ylim([0, 1.05])

plt.suptitle('Curvas de Aprendizado – F-measure (treino e teste)', fontsize=12)
plt.tight_layout()
plt.savefig('curvas_aprendizado_q2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva: curvas_aprendizado_q2.png')
